In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, random_split, Subset
import matplotlib.pyplot as plt
from PIL import Image
import os

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

classes = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']

Using: cuda


In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)
])

data = datasets.CIFAR10('./data', train=True, download=True,
                        transform=transform)

data = Subset(data, range(500))
train, val = random_split(data, [400, 100])

train_loader = DataLoader(train, batch_size=64, shuffle=True)
val_loader = DataLoader(val, batch_size=64)

100.0%


In [4]:
def get_model(name):
    if name == "vgg":
        model = models.vgg16(weights="DEFAULT")
        model.classifier[6] = nn.Linear(4096, 10)

    elif name == "resnet":
        model = models.resnet18(weights="DEFAULT")
        model.fc = nn.Linear(model.fc.in_features, 10)

    else:
        model = models.googlenet(weights="DEFAULT", aux_logits=True)
        model.fc = nn.Linear(model.fc.in_features, 10)

    return model.to(device)

In [5]:
def train(model, epochs=5):
    optimizer = optim.Adam(model.parameters(), lr=0.0001)
    loss_fn = nn.CrossEntropyLoss()
    acc = []

    for epoch in range(epochs):
        model.train()

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            optimizer.step()

        model.eval()
        correct = total = 0

        with torch.no_grad():
            for x, y in val_loader:
                out = model(x.to(device))
                correct += (out.argmax(1) == y.to(device)).sum().item()
                total += len(y)

        accuracy = 100 * correct / total
        acc.append(accuracy)

        print(f"Epoch {epoch+1}: {accuracy:.2f}%")

    return acc

In [ ]:
results = {}

for name in ["vgg", "resnet", "googlenet"]:
    print("\nTraining", name.upper())
    results[name] = train(get_model(name))


Training VGG


0.1%

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\Nishita/.cache\torch\hub\checkpoints\vgg16-397923af.pth


100.0%


Epoch 1: 32.00%
Epoch 2: 66.00%
Epoch 3: 64.00%
Epoch 4: 67.00%
Epoch 5: 77.00%

Training RESNET
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Nishita/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


Epoch 1: 52.00%
Epoch 2: 66.00%
Epoch 3: 71.00%
Epoch 4: 76.00%
Epoch 5: 80.00%

Training GOOGLENET
Downloading: "https://download.pytorch.org/models/googlenet-1378be20.pth" to C:\Users\Nishita/.cache\torch\hub\checkpoints\googlenet-1378be20.pth


76.4%

In [ ]:
for name, acc in results.items():
    plt.plot(acc, label=name.upper())

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy (%)")
plt.title("VGG vs ResNet vs GoogLeNet")
plt.legend()
plt.grid()
plt.show()

In [ ]:
image = transform(Image.open("download.jpeg").convert("RGB"))
image = image.unsqueeze(0).to(device)

for name in ["vgg", "resnet", "googlenet"]:
    model = get_model(name)
    model.eval()

    with torch.no_grad():
        pred = model(image).argmax(1).item()

    print(name.upper(), "=>", classes[pred])